# Chapter 28 Walkthrough: PyTorch Core — Tensors, Autograd, `nn.Module`, Datasets & Training Loops

**Book Reference:** Chapter 28, Sections 28.1 – 28.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. Every tabular deep-learning experiment runs deterministically on CPU/GPU/MPS using the bundled `mch_survey_sample` dataset.

In [ ]:
# @title Step 0: Click Play (▶) to Load PyTorch & MCH Survey Data
!pip install -q sharukh-tutor
import os
import tempfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
import sharukh_tutor as tutor

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
torch.manual_seed(42)

mch_raw = tutor.load_data("mch_survey_sample", as_pandas=True)
mch = mch_raw.copy()
mch["residence"] = (
    mch["residence"].astype(str).str.strip().str.title()
)
mch["anc_visits"] = mch["anc_visits"].replace(999, np.nan)
mch["is_urban"] = (mch["residence"] == "Urban").astype(int)
wealth_map = {
    "Poorest": 1, "Poorer": 2, "Middle": 3,
    "Richer": 4, "Richest": 5,
}
mch["wealth_num"] = mch["wealth_quintile"].map(wealth_map)
feat_cols = [
    "maternal_age_years", "anc_visits", "wealth_num", "is_urban"
]
clean = mch[["maternal_anemia", "child_stunted"] + feat_cols].dropna()
print(f"PyTorch: {torch.__version__} | Active device: {device}")
print(f"Complete MCH rows: {len(clean)} | Features: {len(feat_cols)}")


## Section 28.3.1 — Tensors vs. NumPy, `float32`, Aliasing & Device Portability

In [ ]:
# NumPy defaults to float64; PyTorch layers expect float32
arr_f64 = np.array([[24.0, 4.0], [31.0, 2.0]])
arr_f32 = arr_f64.astype(np.float32)

t_alias = torch.from_numpy(arr_f32)            # shares memory!
t_copy = torch.tensor(arr_f32, dtype=torch.float32)  # safe copy

arr_f32[0, 0] = 999.0
print(f"NumPy default dtype: {arr_f64.dtype} | PyTorch: {t_copy.dtype}")
print(f"from_numpy (aliased) [0,0]: {t_alias[0, 0].item():.1f}")
print(f"torch.tensor (safe)  [0,0]: {t_copy[0, 0].item():.1f}")

t_dev = t_copy.to(device)
print(f"Shape: {tuple(t_dev.shape)} | Device: {t_dev.device}")


## Section 28.3.2 — Dynamic Autograd Graph & the `.grad.zero_()` Accumulation Trap

In [ ]:
# y = 3x^2 - 4x + 5  =>  dy/dx = 6x - 4  (at x=2: y=9, dy/dx=8)
x = torch.tensor(2.0, requires_grad=True)
y = 3.0 * x**2 - 4.0 * x + 5.0
print(f"Forward y={y.item():.1f} | grad_fn={type(y.grad_fn).__name__}")

y.backward()
print(f"After 1st backward()       -> x.grad = {x.grad.item():.1f}")

# Trap: PyTorch ACCUMULATES gradients unless you zero them!
y2 = 3.0 * x**2 - 4.0 * x + 5.0
y2.backward()
print(f"After 2nd (NO zero_grad!)  -> x.grad = {x.grad.item():.1f}")

x.grad.zero_()
y3 = 3.0 * x**2 - 4.0 * x + 5.0
y3.backward()
print(f"After x.grad.zero_() + 3rd -> x.grad = {x.grad.item():.1f}")


## Section 28.3.3 — Custom `Dataset` & Mini-Batch `DataLoader` on `mch_survey_sample`

In [ ]:
X_all = clean[feat_cols].to_numpy(dtype=np.float32)
y_all = clean["child_stunted"].to_numpy(dtype=np.float32)
X_tr, X_va, y_tr, y_va = train_test_split(
    X_all, y_all, test_size=0.25, stratify=y_all, random_state=42
)

# Fit StandardScaler strictly on X_train to prevent leakage
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_va_s = scaler.transform(X_va)

class ClinicalTabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

torch.manual_seed(42)
train_ds = ClinicalTabularDataset(X_tr_s, y_tr)
val_ds = ClinicalTabularDataset(X_va_s, y_va)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

xb, yb = next(iter(train_loader))
print(f"Train n={len(train_ds)} | Val n={len(val_ds)}")
print(
    f"First batch X: {tuple(xb.shape)} ({xb.dtype}) | "
    f"y: {tuple(yb.shape)} ({yb.dtype})"
)


## Section 28.3.4 — Defining `nn.Module` (`ClinicalRiskMLP`) & Pairing Raw Logits with `BCEWithLogitsLoss`

In [ ]:
class ClinicalRiskMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        # Return raw logits — NO sigmoid! BCEWithLogitsLoss fuses it.
        return self.net(x)

torch.manual_seed(42)
model = ClinicalRiskMLP(in_dim=4, hidden_dim=16).to(device)
n_params = sum(p.numel() for p in model.parameters())
logits_0 = model(xb.to(device))
criterion = nn.BCEWithLogitsLoss()
init_loss = criterion(logits_0, yb.to(device)).item()
print(f"Total trainable parameters: {n_params}")
print(
    f"Logits shape: {tuple(logits_0.shape)} | "
    f"Initial batch BCE loss: {init_loss:.5f}"
)


## Section 28.3.5 — The Canonical 5-Step PyTorch Training Loop

In [ ]:
torch.manual_seed(42)
model = ClinicalRiskMLP(in_dim=4, hidden_dim=16).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)

for epoch in range(1, 31):
    model.train()
    tr_sum, tr_n = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()          # 1. Clear stale gradients
        logits = model(xb)             # 2. Forward pass (raw logits)
        loss = criterion(logits, yb)   # 3. Compute scalar BCE loss
        loss.backward()                # 4. Backprop via autograd
        optimizer.step()               # 5. Update weights
        tr_sum += loss.item() * len(xb)
        tr_n += len(xb)
    tr_loss = tr_sum / tr_n

    if epoch in (1, 10, 20, 30):
        model.eval()
        va_sum, va_n, p_list, y_list = 0.0, 0, [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                lg = model(xb)
                va_sum += criterion(lg, yb).item() * len(xb)
                va_n += len(xb)
                p_list.append(torch.sigmoid(lg).cpu().numpy().ravel())
                y_list.append(yb.cpu().numpy().ravel())
        auc = roc_auc_score(
            np.concatenate(y_list), np.concatenate(p_list)
        )
        print(
            f"Epoch {epoch:2d} | train_loss={tr_loss:.4f} | "
            f"val_loss={va_sum / va_n:.4f} | val_auc={auc:.4f}"
        )


## Section 28.3.6 — Inference Mode (`eval()` + `torch.no_grad()`) & Safe `state_dict` Checkpointing

In [ ]:
X_sample = torch.tensor(X_va_s[:8], dtype=torch.float32).to(device)
with tempfile.TemporaryDirectory() as tmpdir:
    ckpt_path = os.path.join(tmpdir, "stunting_mlp.pt")
    torch.save(model.state_dict(), ckpt_path)
    file_bytes = os.path.getsize(ckpt_path)

    reloaded = ClinicalRiskMLP(in_dim=4, hidden_dim=16).to(device)
    state = torch.load(
        ckpt_path, map_location=device, weights_only=True
    )
    reloaded.load_state_dict(state)

    model.eval()
    reloaded.eval()
    with torch.no_grad():
        p_orig = torch.sigmoid(model(X_sample))
        p_load = torch.sigmoid(reloaded(X_sample))
    exact = bool(torch.allclose(p_orig, p_load, atol=1e-6))
    print(f"Checkpoint size: {file_bytes} bytes | Match: {exact}")
    print(f"First 4 probs: {np.round(p_load.cpu().numpy().ravel()[:4], 4)}")


## Section 28.3.7 — Break It On Purpose! (PyTorch Error Anatomy)

In [ ]:
# Intentional Bug #1: float64 tensor fed into a float32 nn.Linear layer
layer = nn.Linear(4, 1)
x_double = torch.tensor([[1.0, 2.0, 3.0, 4.0]], dtype=torch.float64)
layer(x_double)


In [ ]:
# Intentional Bug #2: Feature dimension mismatch (3 cols into 4-in Linear)
layer4 = nn.Linear(4, 16)
x_wrong = torch.randn(8, 3)
layer4(x_wrong)


In [ ]:
# Intentional Bug #3: Calling .backward() twice on a freed graph
w = torch.tensor(3.0, requires_grad=True)
loss = (w - 1.0) ** 2
loss.backward()
loss.backward()


## Section 28.4 — Stage 4 Blueprint: End-to-End Reproducible PyTorch Clinical Pipeline

In [ ]:
def train_and_checkpoint_mch_mlp(
    raw_df, target="child_stunted", epochs=30, seed=42
):
    """Leak-free PyTorch MLP training + weights_only checkpoint audit."""
    torch.manual_seed(seed)
    df = raw_df.copy()
    df["residence"] = df["residence"].astype(str).str.strip().str.title()
    df["anc_visits"] = df["anc_visits"].replace(999, np.nan)
    df["is_urban"] = (df["residence"] == "Urban").astype(int)
    w_map = {
        "Poorest": 1, "Poorer": 2, "Middle": 3,
        "Richer": 4, "Richest": 5,
    }
    df["wealth_num"] = df["wealth_quintile"].map(w_map)
    cols = ["maternal_age_years", "anc_visits", "wealth_num", "is_urban"]
    sub = df[[target] + cols].dropna()
    X_tr, X_va, y_tr, y_va = train_test_split(
        sub[cols].to_numpy(dtype=np.float32),
        sub[target].to_numpy(dtype=np.float32),
        test_size=0.25, stratify=sub[target], random_state=seed,
    )
    sc = StandardScaler()
    tr_ld = DataLoader(
        ClinicalTabularDataset(sc.fit_transform(X_tr), y_tr),
        batch_size=32, shuffle=True,
    )
    va_ld = DataLoader(
        ClinicalTabularDataset(sc.transform(X_va), y_va),
        batch_size=32, shuffle=False,
    )
    net = ClinicalRiskMLP(in_dim=4, hidden_dim=16)
    crit = nn.BCEWithLogitsLoss()
    opt = torch.optim.Adam(net.parameters(), lr=0.02)
    for _ in range(epochs):
        net.train()
        for xb, yb in tr_ld:
            opt.zero_grad()
            loss = crit(net(xb), yb)
            loss.backward()
            opt.step()
    net.eval()
    va_sum, va_n, ps, ys = 0.0, 0, [], []
    with torch.no_grad():
        for xb, yb in va_ld:
            lg = net(xb)
            va_sum += crit(lg, yb).item() * len(xb)
            va_n += len(xb)
            ps.append(torch.sigmoid(lg).numpy().ravel())
            ys.append(yb.numpy().ravel())
    auc = round(
        float(roc_auc_score(np.concatenate(ys), np.concatenate(ps))), 4
    )
    with tempfile.TemporaryDirectory() as tmpdir:
        p = os.path.join(tmpdir, f"{target}.pt")
        torch.save(net.state_dict(), p)
        fresh = ClinicalRiskMLP(in_dim=4, hidden_dim=16)
        fresh.load_state_dict(torch.load(p, weights_only=True))
        fresh.eval()
        x0 = torch.tensor(sc.transform(X_va[:8]), dtype=torch.float32)
        with torch.no_grad():
            ok = bool(torch.allclose(net(x0), fresh(x0), atol=1e-6))
        ckpt_bytes = int(os.path.getsize(p))
    return {
        "val_loss": round(va_sum / va_n, 4),
        "val_auc": auc,
        "ckpt_bytes": ckpt_bytes,
        "ckpt_match": ok,
    }

for label, col in [
    ("Stunting", "child_stunted"),
    ("Anemia", "maternal_anemia"),
]:
    rep = train_and_checkpoint_mch_mlp(mch_raw, target=col)
    print(
        f"{label:8s} | val_loss={rep['val_loss']:.4f} | "
        f"val_auc={rep['val_auc']:.4f} | "
        f"ckpt={rep['ckpt_bytes']}B (match={rep['ckpt_match']})"
    )
